In [2]:
#!/usr/bin/env python
# coding: utf-8

"""
INITIAL BALANCE PREDICTOR ANALYSIS
Goal: What variables can predict IB direction (9:30-10:30am)?
Analyzes: 1) Individual features, 2) 2-feature combos, 3) 3-feature combos, 4) 4-feature combos
UPDATED: Includes Asia/London positional relationships + separate top 100 bullish/bearish results
"""

import pandas as pd
import numpy as np
from datetime import datetime
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

# Adjust the index to group by NY trading day
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

print("="*100)
print("INITIAL BALANCE DIRECTION PREDICTOR")
print("="*100)
print(f"Analyzing {len(dates)-1} trading days from {start_date.date()} to {end_date.date()}")
print("\nTarget: IB Direction (9:30-10:30am) - Bullish or Bearish")
print("\nPredictors available BEFORE or AT START of IB:")
print("  ✓ Floor Pivots (from previous day)")
print("  ✓ Overnight session (6pm-9:30am)")
print("  ✓ Asia, London, Globex sessions")
print("  ✓ Asia/London positional relationships")
print("  ✓ Pre-market ORBs")
print("  ✓ NY 5-min ORB (9:30-9:35am) - direction & first break")
print("  ✓ Gap analysis")
print("  ✓ Previous day/week stats")
print("  ✓ Day of week")
print()

# ============================================================
# FLOOR PIVOT FUNCTIONS
# ============================================================

def calculate_floor_pivots(high, low, close):
    """Calculate floor pivot levels"""
    pp = (high + low + close) / 3
    
    r1 = (2 * pp) - low
    r2 = pp + (high - low)
    r3 = high + 2 * (pp - low)
    
    s1 = (2 * pp) - high
    s2 = pp - (high - low)
    s3 = low - 2 * (high - pp)
    
    # Golden Zone (0.5-0.618 of previous day range)
    pivot_range = high - low
    gz_low = low + pivot_range * 0.5
    gz_high = low + pivot_range * 0.618
    
    return {
        'PP': pp, 'R1': r1, 'R2': r2, 'R3': r3,
        'S1': s1, 'S2': s2, 'S3': s3,
        'GZ_LOW': gz_low, 'GZ_HIGH': gz_high
    }

def get_pivot_position(price, pivots, threshold_pct=0.1):
    """Determine position relative to floor pivots"""
    threshold = pivots['PP'] * (threshold_pct / 100)
    
    if abs(price - pivots['PP']) <= threshold:
        return 'at_pp'
    elif pivots['GZ_LOW'] <= price <= pivots['GZ_HIGH']:
        return 'in_gz'
    elif price >= pivots['R3']:
        return 'above_r3'
    elif price >= pivots['R2']:
        return 'r2_to_r3'
    elif price >= pivots['R1']:
        return 'r1_to_r2'
    elif price > pivots['GZ_HIGH']:
        return 'gz_to_r1'
    elif price > pivots['PP']:
        return 'pp_to_gz'
    elif price <= pivots['S3']:
        return 'below_s3'
    elif price <= pivots['S2']:
        return 's3_to_s2'
    elif price <= pivots['S1']:
        return 's2_to_s1'
    elif price < pivots['GZ_LOW']:
        return 's1_to_gz'
    else:
        return 'gz_to_pp'

def check_level_touch(data, level_price, threshold_pct=0.05):
    """Check if a price level was touched in given data"""
    if data.empty:
        return False
    threshold = level_price * (threshold_pct / 100)
    for _, row in data.iterrows():
        if row['Low'] - threshold <= level_price <= row['High'] + threshold:
            return True
    return False

# ============================================================
# BUILD DATASET WITH IB AS TARGET
# ============================================================

ib_prediction_data = []
skipped_count = 0

print("Building dataset...")
for idx, date in enumerate(dates[1:], 1):
    if idx % 500 == 0:
        print(f"  Processing day {idx}...")
    
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define trading day boundaries
    ny_trading_day_start = actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    ny_trading_day_end = actual_date.replace(hour=17, minute=0, second=0)
    
    full_day_data = df.loc[ny_trading_day_start:ny_trading_day_end]
    if full_day_data.empty:
        skipped_count += 1
        continue
    
    # ========================================
    # TARGET: INITIAL BALANCE (9:30-10:30am)
    # ========================================
    ib_start = actual_date.replace(hour=9, minute=30, second=0)
    ib_end = actual_date.replace(hour=10, minute=30, second=0)
    ib_data = df.loc[ib_start:ib_end]
    
    if ib_data.empty:
        skipped_count += 1
        continue
    
    ib_open = ib_data['Open'].iloc[0]
    ib_close = ib_data['Close'].iloc[-1]
    ib_high = ib_data['High'].max()
    ib_low = ib_data['Low'].min()
    ib_range = ib_high - ib_low
    
    # Skip if flat or invalid
    if pd.isna([ib_open, ib_close]).any() or ib_open == ib_close or ib_range == 0:
        skipped_count += 1
        continue
    
    # IB DIRECTION (TARGET)
    ib_direction = 'bullish' if ib_close > ib_open else 'bearish'
    
    # ========================================
    # PREVIOUS DAY DATA
    # ========================================
    prev_date = date - pd.Timedelta(days=1)
    prev_actual_date = prev_date + pd.Timedelta(hours=7)
    prev_day_start = prev_actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    prev_day_end = prev_actual_date.replace(hour=17, minute=0, second=0)
    prev_day_data = df.loc[prev_day_start:prev_day_end]
    
    if not prev_day_data.empty:
        prev_day_high = prev_day_data['High'].max()
        prev_day_low = prev_day_data['Low'].min()
        prev_day_open = prev_day_data['Open'].iloc[0]
        prev_day_close = prev_day_data['Close'].iloc[-1]
        prev_day_direction = 'bullish' if prev_day_close > prev_day_open else 'bearish'
        prev_day_range = prev_day_high - prev_day_low
    else:
        prev_day_high = prev_day_low = prev_day_open = prev_day_close = None
        prev_day_direction = None
        prev_day_range = None
    
    # ========================================
    # PREVIOUS WEEK DATA
    # ========================================
    current_day_index = list(dates).index(date)
    if current_day_index >= 5:
        prev_week_dates = dates[current_day_index-5:current_day_index]
        prev_week_data_list = []
        
        for pwd in prev_week_dates:
            pwd_actual = pwd + pd.Timedelta(hours=7)
            pwd_start = pwd_actual.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
            pwd_end = pwd_actual.replace(hour=17, minute=0, second=0)
            pwd_data = df.loc[pwd_start:pwd_end]
            if not pwd_data.empty:
                prev_week_data_list.append(pwd_data)
        
        if prev_week_data_list:
            prev_week_combined = pd.concat(prev_week_data_list)
            prev_week_high = prev_week_combined['High'].max()
            prev_week_low = prev_week_combined['Low'].min()
        else:
            prev_week_high = prev_week_low = None
    else:
        prev_week_high = prev_week_low = None
    
    # ========================================
    # FLOOR PIVOTS (from previous day)
    # ========================================
    if prev_day_high and prev_day_low and prev_day_close:
        floor_pivots = calculate_floor_pivots(prev_day_high, prev_day_low, prev_day_close)
    else:
        floor_pivots = None
    
    # ========================================
    # OVERNIGHT SESSION (6pm-9:30am)
    # ========================================
    overnight_start = ny_trading_day_start
    overnight_end = actual_date.replace(hour=9, minute=30, second=0)
    overnight_data = df.loc[overnight_start:overnight_end]
    
    overnight_direction = None
    overnight_high = overnight_low = overnight_open = overnight_close = None
    overnight_range = None
    gap_type = None
    gap_size = None
    overnight_touched_prev_day_high = False
    overnight_touched_prev_day_low = False
    overnight_touched_prev_week_high = False
    overnight_touched_prev_week_low = False
    
    # Pivot touches
    overnight_touched_r3 = overnight_touched_r2 = overnight_touched_r1 = False
    overnight_touched_pp = overnight_touched_s1 = overnight_touched_s2 = False
    overnight_touched_s3 = overnight_touched_gz = False
    overnight_highest_pivot = overnight_lowest_pivot = None
    
    ny_open_pivot_position = None
    
    if not overnight_data.empty:
        overnight_high = overnight_data['High'].max()
        overnight_low = overnight_data['Low'].min()
        overnight_open = overnight_data['Open'].iloc[0]
        overnight_close = overnight_data['Close'].iloc[-1]
        overnight_direction = 'bullish' if overnight_close > overnight_open else 'bearish'
        overnight_range = overnight_high - overnight_low
        
        # Gap analysis
        if prev_day_close:
            gap_size = overnight_open - prev_day_close
            if gap_size > prev_day_range * 0.002:
                gap_type = 'gap_up'
            elif gap_size < -prev_day_range * 0.002:
                gap_type = 'gap_down'
            else:
                gap_type = 'no_gap'
        
        # Level touches overnight
        overnight_touched_prev_day_high = check_level_touch(overnight_data, prev_day_high) if prev_day_high else False
        overnight_touched_prev_day_low = check_level_touch(overnight_data, prev_day_low) if prev_day_low else False
        overnight_touched_prev_week_high = check_level_touch(overnight_data, prev_week_high) if prev_week_high else False
        overnight_touched_prev_week_low = check_level_touch(overnight_data, prev_week_low) if prev_week_low else False
        
        # Floor pivot touches overnight
        if floor_pivots:
            overnight_touched_r3 = check_level_touch(overnight_data, floor_pivots['R3'])
            overnight_touched_r2 = check_level_touch(overnight_data, floor_pivots['R2'])
            overnight_touched_r1 = check_level_touch(overnight_data, floor_pivots['R1'])
            overnight_touched_pp = check_level_touch(overnight_data, floor_pivots['PP'])
            overnight_touched_s1 = check_level_touch(overnight_data, floor_pivots['S1'])
            overnight_touched_s2 = check_level_touch(overnight_data, floor_pivots['S2'])
            overnight_touched_s3 = check_level_touch(overnight_data, floor_pivots['S3'])
            overnight_touched_gz = (overnight_data['Low'].min() <= floor_pivots['GZ_HIGH'] and 
                                   overnight_data['High'].max() >= floor_pivots['GZ_LOW'])
            
            # Highest and lowest pivot touched
            pivot_levels = {'R3': floor_pivots['R3'], 'R2': floor_pivots['R2'], 'R1': floor_pivots['R1'],
                          'PP': floor_pivots['PP'], 'S1': floor_pivots['S1'], 'S2': floor_pivots['S2'], 
                          'S3': floor_pivots['S3']}
            touched_pivots = [k for k, v in pivot_levels.items() if check_level_touch(overnight_data, v)]
            
            if touched_pivots:
                pivot_order = ['R3', 'R2', 'R1', 'PP', 'S1', 'S2', 'S3']
                touched_in_order = [p for p in pivot_order if p in touched_pivots]
                overnight_highest_pivot = touched_in_order[0] if touched_in_order else None
                overnight_lowest_pivot = touched_in_order[-1] if touched_in_order else None
            
            # NY open position relative to floor pivots
            ny_open_pivot_position = get_pivot_position(overnight_close, floor_pivots)
    
    # ========================================
    # ASIA SESSION (6pm-2am)
    # ========================================
    asia_start = ny_trading_day_start
    asia_end = actual_date.replace(hour=2, minute=0, second=0)
    asia_data = df.loc[asia_start:asia_end]
    
    asia_direction = None
    asia_range = None
    asia_high = asia_low = None
    if not asia_data.empty:
        asia_open = asia_data['Open'].iloc[0]
        asia_close = asia_data['Close'].iloc[-1]
        asia_high = asia_data['High'].max()
        asia_low = asia_data['Low'].min()
        asia_direction = 'bullish' if asia_close > asia_open else 'bearish'
        asia_range = asia_high - asia_low
    
    # ========================================
    # LONDON SESSION (2am-8am)
    # ========================================
    london_start = actual_date.replace(hour=2, minute=0, second=0)
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    london_data = df.loc[london_start:london_end]
    
    london_direction = None
    london_range = None
    london_high = london_low = None
    if not london_data.empty:
        london_open = london_data['Open'].iloc[0]
        london_close = london_data['Close'].iloc[-1]
        london_high = london_data['High'].max()
        london_low = london_data['Low'].min()
        london_direction = 'bullish' if london_close > london_open else 'bearish'
        london_range = london_high - london_low
    
    # ========================================
    # ASIA/LONDON POSITIONAL RELATIONSHIP
    # ========================================
    asia_london_position = None
    if asia_high is not None and asia_low is not None and london_high is not None and london_low is not None:
        # Check for engulfment and partial breaks
        if london_low >= asia_low and london_high <= asia_high:
            asia_london_position = 'asia_engulfs_london'
        elif asia_low >= london_low and asia_high <= london_high:
            asia_london_position = 'london_engulfs_asia'
        elif london_high > asia_high and london_low >= asia_low:
            asia_london_position = 'london_partial_up'
        elif london_low < asia_low and london_high <= asia_high:
            asia_london_position = 'london_partial_down'
        elif london_high > asia_high and london_low < asia_low:
            asia_london_position = 'london_breaks_both'
        else:
            asia_london_position = 'overlapping'
    
    # ========================================
    # GLOBEX SESSION (6pm-9:30am) - same as overnight
    # ========================================
    globex_direction = overnight_direction
    
    # ========================================
    # PRE-MARKET ORBs
    # ========================================
    # Globex ORB (first 30 min: 6pm-6:30pm)
    globex_orb_start = ny_trading_day_start
    globex_orb_end = ny_trading_day_start + pd.Timedelta(minutes=30)
    globex_orb_data = df.loc[globex_orb_start:globex_orb_end]
    
    globex_orb_direction = None
    if not globex_orb_data.empty and len(globex_orb_data) >= 5:
        globex_orb_open = globex_orb_data['Open'].iloc[0]
        globex_orb_close = globex_orb_data['Close'].iloc[-1]
        globex_orb_direction = 'bullish' if globex_orb_close > globex_orb_open else 'bearish'
    
    # London ORB (first 30 min: 2am-2:30am)
    london_orb_start = actual_date.replace(hour=2, minute=0, second=0)
    london_orb_end = actual_date.replace(hour=2, minute=30, second=0)
    london_orb_data = df.loc[london_orb_start:london_orb_end]
    
    london_orb_direction = None
    if not london_orb_data.empty and len(london_orb_data) >= 5:
        london_orb_open = london_orb_data['Open'].iloc[0]
        london_orb_close = london_orb_data['Close'].iloc[-1]
        london_orb_direction = 'bullish' if london_orb_close > london_orb_open else 'bearish'
    
    # Asia ORB (first 30 min: 6pm-6:30pm) - same as globex
    asia_orb_direction = globex_orb_direction
    
    # Pre-market ORB (8am-9:30am - last 90 min before NY open)
    premarket_orb_start = actual_date.replace(hour=8, minute=0, second=0)
    premarket_orb_end = actual_date.replace(hour=9, minute=30, second=0)
    premarket_orb_data = df.loc[premarket_orb_start:premarket_orb_end]
    
    premarket_orb_direction = None
    if not premarket_orb_data.empty:
        premarket_orb_open = premarket_orb_data['Open'].iloc[0]
        premarket_orb_close = premarket_orb_data['Close'].iloc[-1]
        premarket_orb_direction = 'bullish' if premarket_orb_close > premarket_orb_open else 'bearish'
    
    # ========================================
    # SESSION PATTERN
    # ========================================
    session_pattern = None
    if asia_direction and london_direction:
        if asia_direction == 'bullish' and london_direction == 'bullish':
            session_pattern = 'bull_bull'
        elif asia_direction == 'bullish' and london_direction == 'bearish':
            session_pattern = 'bull_bear'
        elif asia_direction == 'bearish' and london_direction == 'bullish':
            session_pattern = 'bear_bull'
        else:
            session_pattern = 'bear_bear'
    
    # ========================================
    # NY OPEN VS PREVIOUS DAY
    # ========================================
    ny_open_vs_prev_day = None
    if overnight_close and prev_day_high and prev_day_low:
        if overnight_close > prev_day_high:
            ny_open_vs_prev_day = 'above_prev_high'
        elif overnight_close < prev_day_low:
            ny_open_vs_prev_day = 'below_prev_low'
        else:
            ny_open_vs_prev_day = 'inside_prev_range'
    
    # ========================================
    # NY 5-MINUTE ORB (9:30-9:35am)
    # ========================================
    ny_orb_start = actual_date.replace(hour=9, minute=30, second=0)
    ny_orb_end = actual_date.replace(hour=9, minute=35, second=0)
    ny_orb_data = df.loc[ny_orb_start:ny_orb_end]
    
    ny_orb_direction = None
    ny_orb_range = None
    ny_orb_first_break = None
    
    if not ny_orb_data.empty and len(ny_orb_data) >= 5:
        ny_orb_open = ny_orb_data['Open'].iloc[0]
        ny_orb_close = ny_orb_data['Close'].iloc[-1]
        ny_orb_high = ny_orb_data['High'].max()
        ny_orb_low = ny_orb_data['Low'].min()
        ny_orb_range = ny_orb_high - ny_orb_low
        
        # ORB direction
        if ny_orb_close > ny_orb_open:
            ny_orb_direction = 'bullish'
        elif ny_orb_close < ny_orb_open:
            ny_orb_direction = 'bearish'
        else:
            ny_orb_direction = 'flat'
        
        # Check which extreme was broken first in the rest of IB (9:35-10:30)
        rest_of_ib_start = ny_orb_end
        rest_of_ib_data = df.loc[rest_of_ib_start:ib_end]
        
        if not rest_of_ib_data.empty:
            high_broken = False
            low_broken = False
            
            for idx, row in rest_of_ib_data.iterrows():
                if not high_broken and row['High'] > ny_orb_high:
                    high_broken = True
                    if ny_orb_first_break is None:
                        ny_orb_first_break = 'high'
                
                if not low_broken and row['Low'] < ny_orb_low:
                    low_broken = True
                    if ny_orb_first_break is None:
                        ny_orb_first_break = 'low'
                
                if high_broken and low_broken:
                    break
            
            # If neither broken
            if ny_orb_first_break is None:
                ny_orb_first_break = 'neither'
    
    # ========================================
    # DAY OF WEEK
    # ========================================
    day_of_week = actual_date.strftime('%A')
    
    # ========================================
    # STORE RECORD
    # ========================================
    record = {
        'date': actual_date.date(),
        'day_of_week': day_of_week,
        
        # TARGET
        'ib_direction': ib_direction,
        
        # Previous day
        'prev_day_direction': prev_day_direction,
        'prev_day_range': prev_day_range,
        
        # Overnight
        'overnight_direction': overnight_direction,
        'overnight_range': overnight_range,
        'gap_type': gap_type,
        'gap_size': gap_size,
        
        # Sessions
        'asia_direction': asia_direction,
        'london_direction': london_direction,
        'globex_direction': globex_direction,
        'session_pattern': session_pattern,
        'asia_london_position': asia_london_position,  # NEW
        
        # ORBs
        'globex_orb_direction': globex_orb_direction,
        'london_orb_direction': london_orb_direction,
        'asia_orb_direction': asia_orb_direction,
        'premarket_orb_direction': premarket_orb_direction,
        
        # NY 5-min ORB
        'ny_orb_direction': ny_orb_direction,
        'ny_orb_range': ny_orb_range,
        'ny_orb_first_break': ny_orb_first_break,
        
        # Level touches overnight
        'overnight_touched_prev_day_high': overnight_touched_prev_day_high,
        'overnight_touched_prev_day_low': overnight_touched_prev_day_low,
        'overnight_touched_prev_week_high': overnight_touched_prev_week_high,
        'overnight_touched_prev_week_low': overnight_touched_prev_week_low,
        
        # Floor pivot touches
        'overnight_touched_r3': overnight_touched_r3,
        'overnight_touched_r2': overnight_touched_r2,
        'overnight_touched_r1': overnight_touched_r1,
        'overnight_touched_pp': overnight_touched_pp,
        'overnight_touched_gz': overnight_touched_gz,
        'overnight_touched_s1': overnight_touched_s1,
        'overnight_touched_s2': overnight_touched_s2,
        'overnight_touched_s3': overnight_touched_s3,
        'overnight_highest_pivot': overnight_highest_pivot,
        'overnight_lowest_pivot': overnight_lowest_pivot,
        
        # NY open position
        'ny_open_pivot_position': ny_open_pivot_position,
        'ny_open_vs_prev_day': ny_open_vs_prev_day,
    }
    
    ib_prediction_data.append(record)

# Create DataFrame
ib_df = pd.DataFrame(ib_prediction_data)

print(f"\nDataset built: {len(ib_df)} trading days")
print(f"Skipped: {skipped_count} days (missing data)")
print()


INITIAL BALANCE DIRECTION PREDICTOR
Analyzing 3785 trading days from 2013-01-01 to 2025-12-31

Target: IB Direction (9:30-10:30am) - Bullish or Bearish

Predictors available BEFORE or AT START of IB:
  ✓ Floor Pivots (from previous day)
  ✓ Overnight session (6pm-9:30am)
  ✓ Asia, London, Globex sessions
  ✓ Asia/London positional relationships
  ✓ Pre-market ORBs
  ✓ NY 5-min ORB (9:30-9:35am) - direction & first break
  ✓ Gap analysis
  ✓ Previous day/week stats
  ✓ Day of week

Building dataset...
  Processing day 500...
  Processing day 1000...
  Processing day 1500...
  Processing day 2000...
  Processing day 2500...
  Processing day 3000...
  Processing day 3500...

Dataset built: 3117 trading days
Skipped: 668 days (missing data)



In [6]:
# Save the dataset
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')

# Save

OUTPUT_DIR = r"C:\Users\tammm\NQ and ES analysis"
output_file = os.path.join(OUTPUT_DIR, 'ib_dataset.csv')
ib_df.to_csv(output_file, index=False)


print(f"✅ Saved dataset to: {output_file}")

✅ Saved dataset to: C:\Users\tammm\NQ and ES analysis\ib_dataset.csv


In [12]:
"""
CELL 2: ANALYSIS - Individual + 2/3/4-Feature Combinations
LOADS ib_dataset.csv from disk
Analyzes predictive power of features and their combinations
"""

import pandas as pd
import numpy as np
from datetime import datetime
from itertools import combinations
import warnings
import os
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = r"C:\Users\tammm\NQ and ES analysis"
IB_DATASET_FILE = "ib_dataset.csv"
OUTPUT_DIR = r"C:\Users\tammm\NQ and ES analysis"

ib_dataset_path = os.path.join(DATA_DIR, IB_DATASET_FILE)

# Create output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Created output directory: {OUTPUT_DIR}\n")

# ============================================================
# LOAD THE IB DATASET FROM CSV
# ============================================================

print("="*100)
print("INITIAL BALANCE PREDICTOR ANALYSIS")
print("="*100)

# Check if dataset exists
if not os.path.exists(ib_dataset_path):
    print(f"\n❌ ERROR: Dataset not found at: {ib_dataset_path}")
    print("\nPlease run CELL 1 first to create 'ib_dataset.csv'")
    raise FileNotFoundError(f"Dataset not found: {ib_dataset_path}")

print(f"\n📂 Loading IB dataset from: {ib_dataset_path}")
ib_df = pd.read_csv(ib_dataset_path)
print(f"✅ Loaded {len(ib_df)} trading days")

# Convert date column to datetime if needed
if 'date' in ib_df.columns:
    ib_df['date'] = pd.to_datetime(ib_df['date'])
    print(f"   Date range: {ib_df['date'].min().date()} to {ib_df['date'].max().date()}")

print()

# ============================================================
# BASELINE STATISTICS
# ============================================================

print("="*100)
print("BASELINE STATISTICS")
print("="*100)

total_days = len(ib_df)
bullish_days = (ib_df['ib_direction'] == 'bullish').sum()
bearish_days = total_days - bullish_days
baseline_bullish_pct = (bullish_days / total_days) * 100

print(f"\nTotal days: {total_days}")
print(f"Bullish IB: {bullish_days} ({baseline_bullish_pct:.1f}%)")
print(f"Bearish IB: {bearish_days} ({100-baseline_bullish_pct:.1f}%)")
print(f"\nBaseline: {baseline_bullish_pct:.1f}% bullish")

# Store baseline for later use
BASELINE_BULLISH = baseline_bullish_pct / 100

# ============================================================
# QUALITY FILTERING PARAMETERS
# ============================================================

# Minimum sample sizes for different combo levels
MIN_SAMPLES_1FEATURE = 30    # Individual features
MIN_SAMPLES_2FEATURE = 25    # 2-feature combos
MIN_SAMPLES_3FEATURE = 20    # 3-feature combos  
MIN_SAMPLES_4FEATURE = 15    # 4-feature combos

# Minimum edge % (distance from 50%)
MIN_EDGE_PCT = 5.0  # At least 5% edge (55%+ or 45%- win rate)

# Quality score weights (for ranking)
EDGE_WEIGHT = 0.6        # Weight for edge/confidence
SAMPLE_WEIGHT = 0.4      # Weight for sample size

def calculate_quality_score(edge, sample_size, max_samples):
    """
    Calculate a quality score that balances edge and sample size.
    
    Edge component: normalized edge (0-100% becomes 0-1)
    Sample component: log-scaled sample size (diminishing returns for huge samples)
    """
    edge_score = edge / 50.0  # Normalize edge to 0-2 range (50% edge = 1.0)
    
    # Log scale sample size with diminishing returns
    # 20 samples = ~0.3, 50 samples = ~0.5, 100 samples = ~0.65, 500 samples = ~0.85
    sample_score = np.log(sample_size + 1) / np.log(max_samples + 1)
    
    quality = (EDGE_WEIGHT * edge_score) + (SAMPLE_WEIGHT * sample_score)
    
    return quality

# ============================================================
# FEATURE LIST (available before 9:30am)
# ============================================================

FEATURES = [
    'prev_day_direction',
    'overnight_direction',
    'gap_type',
    'asia_direction',
    'london_direction',
    'globex_direction',
    'session_pattern',
    'asia_london_position',  # NEW
    'globex_orb_direction',
    'london_orb_direction',
    'asia_orb_direction',
    'premarket_orb_direction',
    'ny_orb_direction',
    'ny_orb_first_break',
    'overnight_touched_prev_day_high',
    'overnight_touched_prev_day_low',
    'overnight_touched_prev_week_high',
    'overnight_touched_prev_week_low',
    'overnight_touched_r3',
    'overnight_touched_r2',
    'overnight_touched_r1',
    'overnight_touched_pp',
    'overnight_touched_gz',
    'overnight_touched_s1',
    'overnight_touched_s2',
    'overnight_touched_s3',
    'overnight_highest_pivot',
    'overnight_lowest_pivot',
    'ny_open_pivot_position',
    'ny_open_vs_prev_day',
    'day_of_week',
]

# ============================================================
# PART 1: INDIVIDUAL FEATURE ANALYSIS
# ============================================================

def analyze_single_feature(feature_name, df, baseline=BASELINE_BULLISH, min_samples=MIN_SAMPLES_1FEATURE):
    """Analyze standalone predictive power of a single feature"""
    
    if feature_name not in df.columns:
        return None
    
    feature_df = df[df[feature_name].notna() & df['ib_direction'].notna()]
    
    if len(feature_df) < 50:
        return None
    
    results = []
    max_sample_size = len(feature_df)  # For quality score normalization
    
    for value in feature_df[feature_name].unique():
        subset = feature_df[feature_df[feature_name] == value]
        
        if len(subset) < min_samples:
            continue
        
        bullish_count = len(subset[subset['ib_direction'] == 'bullish'])
        bearish_count = len(subset) - bullish_count
        bullish_pct = bullish_count / len(subset)
        
        # Calculate edge and lift
        edge = abs(bullish_pct - 0.5) * 100
        
        # Filter by minimum edge
        if edge < MIN_EDGE_PCT:
            continue
        
        baseline_for_value = baseline if bullish_pct > 0.5 else (1 - baseline)
        lift = (bullish_pct if bullish_pct > 0.5 else (1 - bullish_pct)) / baseline_for_value
        
        # Calculate quality score
        quality = calculate_quality_score(edge, len(subset), max_sample_size)
        
        results.append({
            'value': value,
            'total_days': len(subset),
            'bullish_count': bullish_count,
            'bearish_count': bearish_count,
            'bullish_pct': bullish_pct * 100,
            'edge': edge,
            'lift': lift,
            'directional_bias': 'bullish' if bullish_pct > 0.5 else 'bearish',
            'quality_score': quality
        })
    
    return results

print("\n\n")
print("="*130)
print("PART 1: INDIVIDUAL FEATURE ANALYSIS")
print("="*130)
print(f"Filters: Min {MIN_SAMPLES_1FEATURE} samples, Min {MIN_EDGE_PCT}% edge")
print("How predictive is each feature BY ITSELF?\n")

feature_summary = []

for feature in FEATURES:
    results = analyze_single_feature(feature, ib_df)
    
    if not results:
        continue
    
    # Find the most predictive value for this feature (by quality score)
    results_df = pd.DataFrame(results)
    best = results_df.loc[results_df['quality_score'].idxmax()]
    
    feature_summary.append({
        'feature': feature,
        'best_value': best['value'],
        'edge': best['edge'],
        'confidence': best['bullish_pct'] if best['directional_bias'] == 'bullish' else 100 - best['bullish_pct'],
        'sample_days': best['total_days'],
        'lift': best['lift'],
        'direction': best['directional_bias'],
        'quality_score': best['quality_score']
    })

summary_df = pd.DataFrame(feature_summary).sort_values('quality_score', ascending=False)

print("Rank | Feature                        | Best Value           | Edge   | Conf   | Lift  | Days | Quality | Dir")
print("-"*130)
for i, row in summary_df.iterrows():
    rank = len(summary_df) - list(summary_df.index).index(i)
    print(f"{rank:4d} | "
          f"{row['feature']:30s} | {str(row['best_value'])[:20]:20s} | "
          f"{row['edge']:5.1f}% | {row['confidence']:5.1f}% | {row['lift']:4.2f}x | "
          f"{row['sample_days']:4d} | {row['quality_score']:7.3f} | {row['direction']:7s}")

# Export individual feature results
output_file = os.path.join(OUTPUT_DIR, 'ib_predictor_individual_features.csv')
summary_df.to_csv(output_file, index=False)

# ============================================================
# PART 2: 2-FEATURE COMBINATIONS
# ============================================================

print("\n\n")
print("="*130)
print("PART 2: 2-FEATURE COMBINATIONS")
print("="*130)
print(f"Filters: Min {MIN_SAMPLES_2FEATURE} samples, Min {MIN_EDGE_PCT}% edge")
print("Which 2-feature combos have the best predictive power?\n")

def analyze_combo(features, values, df, baseline=BASELINE_BULLISH, min_samples=20, max_samples=1000):
    """Analyze a specific combination of feature values"""
    
    # Build filter condition
    condition = True
    for feat, val in zip(features, values):
        if pd.isna(val):
            condition = condition & df[feat].isna()
        else:
            condition = condition & (df[feat] == val)
    
    subset = df[condition]
    
    if len(subset) < min_samples:
        return None
    
    bullish_count = len(subset[subset['ib_direction'] == 'bullish'])
    bearish_count = len(subset) - bullish_count
    bullish_pct = bullish_count / len(subset)
    
    edge = abs(bullish_pct - 0.5) * 100
    
    # Filter by minimum edge
    if edge < MIN_EDGE_PCT:
        return None
    
    baseline_for_value = baseline if bullish_pct > 0.5 else (1 - baseline)
    lift = (bullish_pct if bullish_pct > 0.5 else (1 - bullish_pct)) / baseline_for_value
    
    # Calculate quality score
    quality = calculate_quality_score(edge, len(subset), max_samples)
    
    return {
        'features': ' + '.join(features),
        'values': ' + '.join([str(v) for v in values]),
        'total_days': len(subset),
        'bullish_count': bullish_count,
        'bearish_count': bearish_count,
        'bullish_pct': bullish_pct * 100,
        'edge': edge,
        'lift': lift,
        'directional_bias': 'bullish' if bullish_pct > 0.5 else 'bearish',
        'quality_score': quality
    }

# Analyze 2-feature combinations
combo_2_results = []

print("Analyzing 2-feature combinations (this may take a minute)...")

for feat1, feat2 in combinations(FEATURES, 2):
    # Get all unique value combinations for these two features
    subset = ib_df[[feat1, feat2, 'ib_direction']].dropna()
    
    if len(subset) < MIN_SAMPLES_2FEATURE * 2:
        continue
    
    value_combos = subset.groupby([feat1, feat2]).size()
    
    for (val1, val2), count in value_combos.items():
        if count < MIN_SAMPLES_2FEATURE:
            continue
        
        result = analyze_combo([feat1, feat2], [val1, val2], ib_df, 
                              min_samples=MIN_SAMPLES_2FEATURE,
                              max_samples=len(subset))
        
        if result:
            combo_2_results.append(result)

combo_2_df = pd.DataFrame(combo_2_results).sort_values('quality_score', ascending=False)

print(f"\nFound {len(combo_2_df)} 2-feature combinations meeting quality filters")
print("\nTop 20 2-feature combinations (by quality score):")
print("-"*130)
print("Rank | Features                                | Values                               | Edge   | Conf   | Lift  | Days | Quality | Dir")
print("-"*130)

for idx, (i, row) in enumerate(combo_2_df.head(20).iterrows(), 1):
    print(f"{idx:4d} | "
          f"{row['features'][:40]:40s} | {row['values'][:36]:36s} | "
          f"{row['edge']:5.1f}% | "
          f"{(row['bullish_pct'] if row['directional_bias']=='bullish' else 100-row['bullish_pct']):5.1f}% | "
          f"{row['lift']:4.2f}x | "
          f"{row['total_days']:4d} | {row['quality_score']:7.3f} | {row['directional_bias']:7s}")

# Export 2-feature combo results
combo_2_df.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_2feature_combos.csv'), index=False)

# NEW: Export top 100 bullish and top 100 bearish separately
combo_2_bullish = combo_2_df[combo_2_df['directional_bias'] == 'bullish'].head(100)
combo_2_bearish = combo_2_df[combo_2_df['directional_bias'] == 'bearish'].head(100)

combo_2_bullish.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_2feature_top100_BULLISH.csv'), index=False)
combo_2_bearish.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_2feature_top100_BEARISH.csv'), index=False)

print(f"\nExported: Top 100 BULLISH 2-feature combos (avg {combo_2_bullish['total_days'].mean():.1f} samples)")
print(f"Exported: Top 100 BEARISH 2-feature combos (avg {combo_2_bearish['total_days'].mean():.1f} samples)")

# ============================================================
# PART 3: 3-FEATURE COMBINATIONS
# ============================================================

print("\n\n")
print("="*130)
print("PART 3: 3-FEATURE COMBINATIONS")
print("="*130)
print(f"Filters: Min {MIN_SAMPLES_3FEATURE} samples, Min {MIN_EDGE_PCT}% edge")
print("Which 3-feature combos have the best predictive power?\n")

# For 3-feature combos, we'll be selective to avoid combinatorial explosion
# Use top features from individual analysis
TOP_FEATURES = summary_df.head(10)['feature'].tolist()

combo_3_results = []

print(f"Analyzing 3-feature combinations from top 10 features (this may take a minute)...")

for feat1, feat2, feat3 in combinations(TOP_FEATURES, 3):
    subset = ib_df[[feat1, feat2, feat3, 'ib_direction']].dropna()
    
    if len(subset) < MIN_SAMPLES_3FEATURE * 3:
        continue
    
    value_combos = subset.groupby([feat1, feat2, feat3]).size()
    
    for (val1, val2, val3), count in value_combos.items():
        if count < MIN_SAMPLES_3FEATURE:
            continue
        
        result = analyze_combo([feat1, feat2, feat3], [val1, val2, val3], ib_df,
                              min_samples=MIN_SAMPLES_3FEATURE,
                              max_samples=len(subset))
        
        if result:
            combo_3_results.append(result)

combo_3_df = pd.DataFrame(combo_3_results).sort_values('quality_score', ascending=False)

print(f"\nFound {len(combo_3_df)} 3-feature combinations meeting quality filters")
print("\nTop 20 3-feature combinations (by quality score):")
print("-"*130)
print("Rank | Features                                                    | Edge   | Conf   | Lift  | Days | Quality | Dir")
print("-"*130)

for idx, (i, row) in enumerate(combo_3_df.head(20).iterrows(), 1):
    print(f"{idx:4d} | "
          f"{row['features'][:60]:60s} | "
          f"{row['edge']:5.1f}% | "
          f"{(row['bullish_pct'] if row['directional_bias']=='bullish' else 100-row['bullish_pct']):5.1f}% | "
          f"{row['lift']:4.2f}x | "
          f"{row['total_days']:4d} | {row['quality_score']:7.3f} | {row['directional_bias']:7s}")

# Export 3-feature combo results
combo_3_df.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_3feature_combos.csv'), index=False)

# NEW: Export top 100 bullish and top 100 bearish separately
combo_3_bullish = combo_3_df[combo_3_df['directional_bias'] == 'bullish'].head(100)
combo_3_bearish = combo_3_df[combo_3_df['directional_bias'] == 'bearish'].head(100)

combo_3_bullish.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_3feature_top100_BULLISH.csv'), index=False)
combo_3_bearish.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_3feature_top100_BEARISH.csv'), index=False)

print(f"\nExported: Top 100 BULLISH 3-feature combos (avg {combo_3_bullish['total_days'].mean():.1f} samples)")
print(f"Exported: Top 100 BEARISH 3-feature combos (avg {combo_3_bearish['total_days'].mean():.1f} samples)")

# ============================================================
# PART 4: 4-FEATURE COMBINATIONS
# ============================================================

print("\n\n")
print("="*130)
print("PART 4: 4-FEATURE COMBINATIONS")
print("="*130)
print(f"Filters: Min {MIN_SAMPLES_4FEATURE} samples, Min {MIN_EDGE_PCT}% edge")
print("Which 4-feature combos have the best predictive power?\n")

# For 4-feature combos, use top 8 features to keep runtime reasonable
TOP_FEATURES_4 = summary_df.head(8)['feature'].tolist()

combo_4_results = []

print(f"Analyzing 4-feature combinations from top 8 features (this may take a few minutes)...")

for feat1, feat2, feat3, feat4 in combinations(TOP_FEATURES_4, 4):
    subset = ib_df[[feat1, feat2, feat3, feat4, 'ib_direction']].dropna()
    
    if len(subset) < MIN_SAMPLES_4FEATURE * 4:
        continue
    
    value_combos = subset.groupby([feat1, feat2, feat3, feat4]).size()
    
    for (val1, val2, val3, val4), count in value_combos.items():
        if count < MIN_SAMPLES_4FEATURE:
            continue
        
        result = analyze_combo([feat1, feat2, feat3, feat4], [val1, val2, val3, val4], ib_df,
                              min_samples=MIN_SAMPLES_4FEATURE,
                              max_samples=len(subset))
        
        if result:
            combo_4_results.append(result)

combo_4_df = pd.DataFrame(combo_4_results).sort_values('quality_score', ascending=False)

print(f"\nFound {len(combo_4_df)} 4-feature combinations meeting quality filters")
print("\nTop 20 4-feature combinations (by quality score):")
print("-"*130)
print("Rank | Features                                                                        | Edge   | Conf   | Lift  | Days | Quality | Dir")
print("-"*130)

for idx, (i, row) in enumerate(combo_4_df.head(20).iterrows(), 1):
    print(f"{idx:4d} | "
          f"{row['features'][:80]:80s} | "
          f"{row['edge']:5.1f}% | "
          f"{(row['bullish_pct'] if row['directional_bias']=='bullish' else 100-row['bullish_pct']):5.1f}% | "
          f"{row['lift']:4.2f}x | "
          f"{row['total_days']:4d} | {row['quality_score']:7.3f} | {row['directional_bias']:7s}")

# Export 4-feature combo results
combo_4_df.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_4feature_combos.csv'), index=False)

# NEW: Export top 100 bullish and top 100 bearish separately
combo_4_bullish = combo_4_df[combo_4_df['directional_bias'] == 'bullish'].head(100)
combo_4_bearish = combo_4_df[combo_4_df['directional_bias'] == 'bearish'].head(100)

combo_4_bullish.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_4feature_top100_BULLISH.csv'), index=False)
combo_4_bearish.to_csv(os.path.join(OUTPUT_DIR, 'ib_predictor_4feature_top100_BEARISH.csv'), index=False)

print(f"\nExported: Top 100 BULLISH 4-feature combos (avg {combo_4_bullish['total_days'].mean():.1f} samples)")
print(f"Exported: Top 100 BEARISH 4-feature combos (avg {combo_4_bearish['total_days'].mean():.1f} samples)")

# ============================================================
# SUMMARY AND KEY INSIGHTS
# ============================================================

print("\n\n")
print("="*130)
print("KEY INSIGHTS - WHAT PREDICTS INITIAL BALANCE DIRECTION?")
print("="*130)
print(f"\nQuality Filters Applied:")
print(f"  • Minimum edge: {MIN_EDGE_PCT}%")
print(f"  • Min samples: 1-feature={MIN_SAMPLES_1FEATURE}, 2-feature={MIN_SAMPLES_2FEATURE}, 3-feature={MIN_SAMPLES_3FEATURE}, 4-feature={MIN_SAMPLES_4FEATURE}")
print(f"  • Quality score: {int(EDGE_WEIGHT*100)}% edge weight, {int(SAMPLE_WEIGHT*100)}% sample size weight")

print("\n1. TOP 50 INDIVIDUAL FEATURES (by quality):")
print("-"*80)
for idx, row in summary_df.head(50).iterrows():
    print(f"   {summary_df.index.tolist().index(idx)+1}. {row['feature']:50s} - {row['edge']:.1f}% edge, {row['sample_days']} days")
    print(f"      Best: {row['best_value']} → {row['confidence']:.1f}% {row['direction']} (Quality: {row['quality_score']:.50f})")

print("\n2. TOP 30 TWO-FEATURE COMBINATIONS (by quality):")
print("-"*80)
for idx, row in combo_2_df.head(30).iterrows():
    conf = row['bullish_pct'] if row['directional_bias']=='bullish' else 100-row['bullish_pct']
    print(f"   {combo_2_df.index.tolist().index(idx)+1}. {row['features']}")
    print(f"      When: {row['values']}")
    print(f"      → {conf:.1f}% {row['directional_bias']} ({row['edge']:.1f}% edge, {row['total_days']} days, Q:{row['quality_score']:.30f})")

print("\n3. TOP 30 THREE-FEATURE COMBINATIONS (by quality):")
print("-"*80)
for idx, row in combo_3_df.head(30).iterrows():
    conf = row['bullish_pct'] if row['directional_bias']=='bullish' else 100-row['bullish_pct']
    print(f"   {combo_3_df.index.tolist().index(idx)+1}. {row['features']}")
    print(f"      When: {row['values']}")
    print(f"      → {conf:.1f}% {row['directional_bias']} ({row['edge']:.1f}% edge, {row['total_days']} days, Q:{row['quality_score']:.30f})")

print("\n4. TOP 30 FOUR-FEATURE COMBINATIONS (by quality):")
print("-"*80)
for idx, row in combo_4_df.head(30).iterrows():
    conf = row['bullish_pct'] if row['directional_bias']=='bullish' else 100-row['bullish_pct']
    print(f"   {combo_4_df.index.tolist().index(idx)+1}. {row['features']}")
    print(f"      When: {row['values']}")
    print(f"      → {conf:.1f}% {row['directional_bias']} ({row['edge']:.1f}% edge, {row['total_days']} days, Q:{row['quality_score']:.30f})")

print("\n5. ANALYSIS SUMMARY:")
print("-"*80)
best_single = summary_df.iloc[0]
best_2 = combo_2_df.iloc[0] if len(combo_2_df) > 0 else None
best_3 = combo_3_df.iloc[0] if len(combo_3_df) > 0 else None
best_4 = combo_4_df.iloc[0] if len(combo_4_df) > 0 else None

print(f"   Best single feature:      {best_single['edge']:.1f}% edge ({best_single['sample_days']} samples)")
if best_2 is not None:
    print(f"   Best 2-feature combo:     {best_2['edge']:.1f}% edge ({best_2['total_days']} samples)")
if best_3 is not None:
    print(f"   Best 3-feature combo:     {best_3['edge']:.1f}% edge ({best_3['total_days']} samples)")
if best_4 is not None:
    print(f"   Best 4-feature combo:     {best_4['edge']:.1f}% edge ({best_4['total_days']} samples)")

print("\n" + "="*130)
print("ANALYSIS COMPLETE")
print("="*130)
print(f"\nExported files to '{OUTPUT_DIR}/':")
print(f"  ✓ ib_predictor_individual_features.csv")
print(f"  ✓ ib_predictor_2feature_combos.csv")
print(f"  ✓ ib_predictor_2feature_top100_BULLISH.csv")
print(f"  ✓ ib_predictor_2feature_top100_BEARISH.csv")
print(f"  ✓ ib_predictor_3feature_combos.csv")
print(f"  ✓ ib_predictor_3feature_top100_BULLISH.csv")
print(f"  ✓ ib_predictor_3feature_top100_BEARISH.csv")
print(f"  ✓ ib_predictor_4feature_combos.csv")
print(f"  ✓ ib_predictor_4feature_top100_BULLISH.csv")
print(f"  ✓ ib_predictor_4feature_top100_BEARISH.csv")
print("\n" + "="*130)

INITIAL BALANCE PREDICTOR ANALYSIS

📂 Loading IB dataset from: C:\Users\tammm\NQ and ES analysis\ib_dataset.csv
✅ Loaded 3117 trading days
   Date range: 2013-01-02 to 2025-03-14

BASELINE STATISTICS

Total days: 3117
Bullish IB: 1628 (52.2%)
Bearish IB: 1489 (47.8%)

Baseline: 52.2% bullish



PART 1: INDIVIDUAL FEATURE ANALYSIS
Filters: Min 30 samples, Min 5.0% edge
How predictive is each feature BY ITSELF?

Rank | Feature                        | Best Value           | Edge   | Conf   | Lift  | Days | Quality | Dir
----------------------------------------------------------------------------------------------------------------------------------
   9 | ny_orb_first_break             | high                 |  20.6% |  70.6% | 1.35x | 1630 |   0.615 | bullish
   8 | ny_orb_direction               | bullish              |  18.3% |  68.3% | 1.31x | 1567 |   0.585 | bullish
   7 | day_of_week                    | Monday               |   5.3% |  55.3% | 1.06x |  620 |   0.384 | bullish
   


Found 1599 4-feature combinations meeting quality filters

Top 20 4-feature combinations (by quality score):
----------------------------------------------------------------------------------------------------------------------------------
Rank | Features                                                                        | Edge   | Conf   | Lift  | Days | Quality | Dir
----------------------------------------------------------------------------------------------------------------------------------
   1 | ny_orb_first_break + ny_orb_direction + overnight_touched_s1 + overnight_highest |  44.4% |  94.4% | 1.81x |   18 |   0.688 | bullish
   2 | ny_orb_first_break + ny_orb_direction + day_of_week + ny_open_pivot_position     |  44.1% |  94.1% | 1.80x |   17 |   0.677 | bullish
   3 | ny_orb_first_break + ny_orb_direction + overnight_lowest_pivot + ny_open_pivot_p |  40.9% |  90.9% | 1.74x |   33 |   0.677 | bullish
   4 | ny_orb_first_break + overnight_highest_pivot + overnight_lowes

In [14]:
"""
TOP 50 BULLISH & BEARISH PATTERNS - INITIAL BALANCE DIRECTION
Shows tradeable patterns with baseline-adjusted metrics
"""

import pandas as pd
import numpy as np
import os

# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = r"C:\Users\tammm\NQ and ES analysis"
IB_DATASET_FILE = "ib_dataset.csv"
OUTPUT_DIR = r"C:\Users\tammm\NQ and ES analysis"

ib_dataset_path = os.path.join(DATA_DIR, IB_DATASET_FILE)

# Create output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Created output directory: {OUTPUT_DIR}\n")

# ============================================================
# LOAD ANALYSIS RESULTS
# ============================================================

print("="*130)
print("TOP 50 BULLISH & BEARISH PATTERNS - INITIAL BALANCE DIRECTION")
print("="*130)
print("\nLoading analysis results...")

# Load the combination results from CELL 2
combo_2_path = os.path.join(OUTPUT_DIR, 'ib_predictor_2feature_combos.csv')
combo_3_path = os.path.join(OUTPUT_DIR, 'ib_predictor_3feature_combos.csv')
combo_4_path = os.path.join(OUTPUT_DIR, 'ib_predictor_4feature_combos.csv')

# Check which files exist
available_files = []
if os.path.exists(combo_2_path):
    available_files.append(('2-feature', combo_2_path))
if os.path.exists(combo_3_path):
    available_files.append(('3-feature', combo_3_path))
if os.path.exists(combo_4_path):
    available_files.append(('4-feature', combo_4_path))

if not available_files:
    print("\n❌ ERROR: No combination analysis files found!")
    print("Please run CELL 2 first to generate the analysis results.")
    exit(1)

# Load and combine all available results
all_patterns = []
for name, filepath in available_files:
    df = pd.read_csv(filepath)
    df['pattern_type'] = name
    all_patterns.append(df)
    print(f"✓ Loaded {name} combinations: {len(df):,} patterns")

patterns_df = pd.concat(all_patterns, ignore_index=True)

print(f"\n✓ Total patterns loaded: {len(patterns_df):,}")

# ============================================================
# CALCULATE BASELINE FROM DATA
# ============================================================

# Load the main dataset to get true baseline
ib_dataset_path = os.path.join(DATA_DIR, 'ib_dataset.csv')
if os.path.exists(ib_dataset_path):
    ib_df = pd.read_csv(ib_dataset_path)
    total_days = len(ib_df)
    bullish_days = (ib_df['ib_direction'] == 'bullish').sum()
    BASELINE_BULLISH_PCT = (bullish_days / total_days) * 100
    BASELINE_BEARISH_PCT = 100 - BASELINE_BULLISH_PCT
    print(f"\nBaseline from data:")
    print(f"  Bullish: {BASELINE_BULLISH_PCT:.1f}% ({bullish_days}/{total_days} days)")
    print(f"  Bearish: {BASELINE_BEARISH_PCT:.1f}% ({total_days - bullish_days}/{total_days} days)")
else:
    # Default baseline if dataset not found
    BASELINE_BULLISH_PCT = 50.0
    BASELINE_BEARISH_PCT = 50.0
    print(f"\n⚠️  Using default 50/50 baseline (dataset not found)")

# ============================================================
# FILTER CRITERIA
# ============================================================

MIN_SAMPLE = 50  # Minimum days for tradeable pattern
MIN_CONFIDENCE = 65.0  # Minimum confidence %

print(f"\nFilter criteria:")
print(f"  Minimum sample size: {MIN_SAMPLE} days")
print(f"  Minimum confidence: {MIN_CONFIDENCE}%")

# ============================================================
# CALCULATE ENHANCED METRICS
# ============================================================

def calculate_true_edge(row, baseline_pct):
    """Calculate true edge over baseline"""
    if row['directional_bias'] == 'bullish':
        return row['bullish_pct'] - baseline_pct
    else:
        bearish_pct = 100 - row['bullish_pct']
        return bearish_pct - (100 - baseline_pct)

def calculate_actionability_score(row):
    """Calculate actionability score balancing edge and frequency"""
    edge = row['edge']
    sample = row['total_days']
    
    # Edge component (0-100)
    confidence_score = min(edge * 2, 100)
    
    # Sample size component (0-100) with diminishing returns
    if sample < MIN_SAMPLE:
        freq_score = (sample / MIN_SAMPLE) * 50
    else:
        freq_score = min(50 + (np.log2(sample / MIN_SAMPLE) * 25), 100)
    
    # Weighted combination (60% edge, 40% frequency)
    return round((confidence_score * 0.6) + (freq_score * 0.4), 1)

# Add enhanced metrics
patterns_df['true_edge'] = patterns_df.apply(
    lambda row: calculate_true_edge(row, BASELINE_BULLISH_PCT), axis=1
)
patterns_df['actionability_score'] = patterns_df.apply(calculate_actionability_score, axis=1)

# ============================================================
# FILTER FOR TRADEABLE PATTERNS
# ============================================================

bullish_patterns = patterns_df[
    (patterns_df['directional_bias'] == 'bullish') &
    (patterns_df['total_days'] >= MIN_SAMPLE) &
    (patterns_df['bullish_pct'] >= MIN_CONFIDENCE)
].copy()

bearish_patterns = patterns_df[
    (patterns_df['directional_bias'] == 'bearish') &
    (patterns_df['total_days'] >= MIN_SAMPLE) &
    ((100 - patterns_df['bullish_pct']) >= MIN_CONFIDENCE)
].copy()

# Sort by actionability score
bullish_patterns = bullish_patterns.sort_values('actionability_score', ascending=False)
bearish_patterns = bearish_patterns.sort_values('actionability_score', ascending=False)

print(f"\nTradeable patterns found:")
print(f"  Bullish: {len(bullish_patterns):,} patterns")
print(f"  Bearish: {len(bearish_patterns):,} patterns")

# ============================================================
# DISPLAY TOP 50 BULLISH PATTERNS
# ============================================================

def display_patterns(df, direction, n=50):
    """Display top N patterns with detailed metrics"""
    
    print("\n\n" + "="*130)
    print(f"TOP {n} {direction.upper()} PATTERNS - INITIAL BALANCE DIRECTION")
    print("="*130)
    
    if len(df) == 0:
        print(f"\n⚠️  No tradeable {direction} patterns found with current criteria")
        print(f"Try lowering minimum confidence to 60% or minimum sample to 30 days.")
        return
    
    print(f"\nShowing top {min(n, len(df))} of {len(df):,} tradeable {direction} patterns")
    print(f"Sorted by: Actionability Score (balance of edge and frequency)\n")
    
    for i, (idx, row) in enumerate(df.head(n).iterrows(), 1):
        
        # Prepare metrics
        if direction == 'bullish':
            confidence = row['bullish_pct']
            true_edge = row['true_edge']
            outcome_count = row['bullish_count']
        else:
            confidence = 100 - row['bullish_pct']
            true_edge = row['true_edge']
            outcome_count = row['bearish_count']
        
        # Display header
        print(f"#{i:2d} | Score: {row['actionability_score']:.1f} | "
              f"Conf: {confidence:.1f}% | Edge: {row['edge']:.1f}% | "
              f"True Edge: {true_edge:+.1f}%")
        
        print(f"     Pattern: {row['pattern_type']} | "
              f"Sample: {row['total_days']} days | "
              f"Lift: {row['lift']:.2f}x | "
              f"Quality: {row['quality_score']:.3f}")
        
        print(f"     Conditions: {row['features']}")
        print(f"     Values: {row['values']}")
        
        print(f"     → {direction.upper()} IB: {outcome_count}/{row['total_days']} days "
              f"({confidence:.1f}% accuracy)")
        print()

# Display both sets
display_patterns(bullish_patterns, 'bullish', 50)
display_patterns(bearish_patterns, 'bearish', 50)

# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("\n" + "="*130)
print("SUMMARY STATISTICS")
print("="*130)

print(f"\nBullish Patterns (min {MIN_SAMPLE} days, min {MIN_CONFIDENCE}% conf):")
if len(bullish_patterns) > 0:
    print(f"  Total tradeable: {len(bullish_patterns):,}")
    print(f"  Best confidence: {bullish_patterns['bullish_pct'].max():.1f}%")
    print(f"  Best true edge: {bullish_patterns['true_edge'].max():+.1f}%")
    print(f"  Best score: {bullish_patterns['actionability_score'].max():.1f}")
    print(f"  Average confidence: {bullish_patterns['bullish_pct'].mean():.1f}%")
    print(f"  Average sample: {bullish_patterns['total_days'].mean():.0f} days")
    print(f"  Average lift: {bullish_patterns['lift'].mean():.2f}x")
else:
    print(f"  No tradeable patterns found")

print(f"\nBearish Patterns (min {MIN_SAMPLE} days, min {MIN_CONFIDENCE}% conf):")
if len(bearish_patterns) > 0:
    bearish_conf = 100 - bearish_patterns['bullish_pct']
    print(f"  Total tradeable: {len(bearish_patterns):,}")
    print(f"  Best confidence: {bearish_conf.max():.1f}%")
    print(f"  Best true edge: {bearish_patterns['true_edge'].max():+.1f}%")
    print(f"  Best score: {bearish_patterns['actionability_score'].max():.1f}")
    print(f"  Average confidence: {bearish_conf.mean():.1f}%")
    print(f"  Average sample: {bearish_patterns['total_days'].mean():.0f} days")
    print(f"  Average lift: {bearish_patterns['lift'].mean():.2f}x")
else:
    print(f"  No tradeable patterns found")

# ============================================================
# PATTERN COMPLEXITY BREAKDOWN
# ============================================================

print("\n" + "="*130)
print("PATTERN COMPLEXITY ANALYSIS")
print("="*130)

print("\nBullish Patterns by Complexity:")
if len(bullish_patterns) > 0:
    for ptype in ['2-feature', '3-feature', '4-feature']:
        subset = bullish_patterns[bullish_patterns['pattern_type'] == ptype]
        if len(subset) > 0:
            print(f"  {ptype:12s}: {len(subset):4d} patterns | "
                  f"Avg conf: {subset['bullish_pct'].mean():.1f}% | "
                  f"Avg sample: {subset['total_days'].mean():.0f} days")

print("\nBearish Patterns by Complexity:")
if len(bearish_patterns) > 0:
    for ptype in ['2-feature', '3-feature', '4-feature']:
        subset = bearish_patterns[bearish_patterns['pattern_type'] == ptype]
        if len(subset) > 0:
            bearish_conf = 100 - subset['bullish_pct']
            print(f"  {ptype:12s}: {len(subset):4d} patterns | "
                  f"Avg conf: {bearish_conf.mean():.1f}% | "
                  f"Avg sample: {subset['total_days'].mean():.0f} days")

# ============================================================
# EXPORT RESULTS
# ============================================================

print("\n" + "="*130)
print("EXPORTING RESULTS")
print("="*130)

if len(bullish_patterns) > 0:
    output_file = os.path.join(OUTPUT_DIR, 'tradeable_bullish_ib_top50.csv')
    bullish_patterns.head(50).to_csv(output_file, index=False)
    print(f"\n✓ Exported: tradeable_bullish_ib_top50.csv ({min(50, len(bullish_patterns))} patterns)")

if len(bearish_patterns) > 0:
    output_file = os.path.join(OUTPUT_DIR, 'tradeable_bearish_ib_top50.csv')
    bearish_patterns.head(50).to_csv(output_file, index=False)
    print(f"✓ Exported: tradeable_bearish_ib_top50.csv ({min(50, len(bearish_patterns))} patterns)")

# ============================================================
# KEY INSIGHTS
# ============================================================

print("\n" + "="*130)
print("KEY INSIGHTS")
print("="*130)

print(f"\n1. BASELINE CONTEXT:")
print(f"   Market baseline: {BASELINE_BULLISH_PCT:.1f}% bullish, {BASELINE_BEARISH_PCT:.1f}% bearish")
print(f"   This means the market has a natural {abs(BASELINE_BULLISH_PCT - 50):.1f}% bias")

print(f"\n2. TRUE EDGE CALCULATION:")
print(f"   - Pattern at 70% bullish = {70 - BASELINE_BULLISH_PCT:+.1f}% true edge")
print(f"   - Pattern at 75% bullish = {75 - BASELINE_BULLISH_PCT:+.1f}% true edge")
print(f"   - Pattern at 80% bullish = {80 - BASELINE_BULLISH_PCT:+.1f}% true edge")
print(f"   - Pattern at 70% bearish = {70 - BASELINE_BEARISH_PCT:+.1f}% true edge")

if len(bullish_patterns) > 0 and len(bearish_patterns) > 0:
    print(f"\n3. BULLISH vs BEARISH:")
    ratio = len(bullish_patterns) / len(bearish_patterns)
    if ratio > 2:
        print(f"   ⚠️  Bullish patterns outnumber bearish {ratio:.1f}:1")
        print(f"   This reflects the {BASELINE_BULLISH_PCT:.1f}% bullish baseline")
        print(f"   Bearish setups are rarer but potentially more valuable when they occur")
    elif ratio < 0.5:
        print(f"   ⚠️  Bearish patterns outnumber bullish {1/ratio:.1f}:1")
        print(f"   Unusual given {BASELINE_BULLISH_PCT:.1f}% bullish baseline")
    else:
        print(f"   ✓ Balanced pattern distribution ({ratio:.1f}:1 bullish/bearish ratio)")
elif len(bullish_patterns) == 0:
    print(f"\n3. ⚠️  NO TRADEABLE BULLISH PATTERNS FOUND")
    print(f"   Try lowering minimum confidence to 60% or minimum sample to 30 days")
elif len(bearish_patterns) == 0:
    print(f"\n3. ⚠️  NO TRADEABLE BEARISH PATTERNS FOUND")
    print(f"   This is expected with {BASELINE_BULLISH_PCT:.1f}% bullish baseline")
    print(f"   Try lowering minimum confidence to 60% or minimum sample to 30 days")

print(f"\n4. ACTIONABILITY SCORE:")
print(f"   Score balances:")
print(f"   - 60% Edge (how much better than random)")
print(f"   - 40% Frequency (how often pattern occurs)")
print(f"   Higher score = more tradeable pattern")

print(f"\n5. PATTERN COMPLEXITY:")
print(f"   - 2-feature: Good balance of edge and frequency")
print(f"   - 3-feature: Better edge, lower frequency")
print(f"   - 4-feature: Best edge, lowest frequency")
print(f"   Simple patterns (2-feature) often outperform complex ones in practice")

print("\n" + "="*130)

TOP 50 BULLISH & BEARISH PATTERNS - INITIAL BALANCE DIRECTION

Loading analysis results...
✓ Loaded 2-feature combinations: 902 patterns
✓ Loaded 3-feature combinations: 899 patterns
✓ Loaded 4-feature combinations: 1,599 patterns

✓ Total patterns loaded: 3,400

Baseline from data:
  Bullish: 52.2% (1628/3117 days)
  Bearish: 47.8% (1489/3117 days)

Filter criteria:
  Minimum sample size: 50 days
  Minimum confidence: 65.0%

Tradeable patterns found:
  Bullish: 629 patterns
  Bearish: 430 patterns


TOP 50 BULLISH PATTERNS - INITIAL BALANCE DIRECTION

Showing top 50 of 629 tradeable bullish patterns
Sorted by: Actionability Score (balance of edge and frequency)

# 1 | Score: 73.6 | Conf: 78.5% | Edge: 28.5% | True Edge: +26.3%
     Pattern: 3-feature | Sample: 191 days | Lift: 1.50x | Quality: 0.619
     Conditions: ny_orb_first_break + ny_orb_direction + overnight_highest_pivot
     Values: high + bullish + R1
     → BULLISH IB: 150/191 days (78.5% accuracy)

# 2 | Score: 73.3 | Conf

In [3]:
#!/usr/bin/env python
# coding: utf-8

"""
INITIAL BALANCE - TOP 50 PATTERNS BY COMPLEXITY
Shows: Top 50 bullish and bearish patterns organized by feature count (1, 2, 3 features)
"""

import pandas as pd
import numpy as np
import os

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_DIR = r"C:\Users\tammm\NQ and ES analysis"

# Load smart patterns (or generated pattern files)
print("="*130)
print("INITIAL BALANCE - TOP 50 PATTERNS BY COMPLEXITY")
print("="*130)
print("\nLoading IB pattern analysis results...")

# Try to load from smart_rules file first
smart_file = os.path.join(INPUT_DIR, 'smart_rules_ib.csv')
combo_2_file = os.path.join(INPUT_DIR, 'ib_predictor_2feature_combos.csv')
combo_3_file = os.path.join(INPUT_DIR, 'ib_predictor_3feature_combos.csv')
combo_4_file = os.path.join(INPUT_DIR, 'ib_predictor_4feature_combos.csv')

# Check which file format is available
if os.path.exists(smart_file):
    print(f"✓ Loading from smart_rules_ib.csv")
    df = pd.read_csv(smart_file)
    USE_SMART_FORMAT = True
elif os.path.exists(combo_2_file) or os.path.exists(combo_3_file):
    print(f"✓ Loading from combo files")
    # Load combo files
    all_patterns = []
    if os.path.exists(combo_2_file):
        df2 = pd.read_csv(combo_2_file)
        df2['pattern_type'] = '2-feature'
        df2['feature_count'] = 2
        all_patterns.append(df2)
        print(f"  Loaded 2-feature: {len(df2):,}")
    if os.path.exists(combo_3_file):
        df3 = pd.read_csv(combo_3_file)
        df3['pattern_type'] = '3-feature'
        df3['feature_count'] = 3
        all_patterns.append(df3)
        print(f"  Loaded 3-feature: {len(df3):,}")
    if os.path.exists(combo_4_file):
        df4 = pd.read_csv(combo_4_file)
        df4['pattern_type'] = '4-feature'
        df4['feature_count'] = 4
        all_patterns.append(df4)
        print(f"  Loaded 4-feature: {len(df4):,}")
    
    df = pd.concat(all_patterns, ignore_index=True)
    USE_SMART_FORMAT = False
else:
    print("\n❌ ERROR: No IB pattern files found!")
    print("Looking for either:")
    print(f"  - {smart_file}")
    print(f"  - {combo_2_file}, {combo_3_file}, {combo_4_file}")
    exit(1)

print(f"\n✓ Total patterns loaded: {len(df):,}")

# ============================================================
# CALCULATE BASELINE
# ============================================================

BASELINE_BULLISH = 0.546  # Adjust based on your IB data
BASELINE_BEARISH = 0.454

print(f"\nBaseline:")
print(f"  {BASELINE_BULLISH*100:.1f}% bullish | {BASELINE_BEARISH*100:.1f}% bearish")
print()

# ============================================================
# PREPARE DATA FORMAT
# ============================================================

if USE_SMART_FORMAT:
    # Data is in smart_rules format with 'confidence' as decimal
    # Create outcome column if not exists
    if 'outcome' not in df.columns:
        df['outcome'] = df.apply(lambda x: 'bullish' if x['confidence'] > 0.5 else 'bearish', axis=1)
    
    # Calculate score if not exists
    if 'score' not in df.columns:
        def calc_score(row):
            edge = abs(row['confidence'] - 0.5) * 100
            confidence_score = min(edge * 2, 100)
            sample = row['condition_count']
            if sample < 50:
                freq_score = (sample / 50) * 50
            else:
                freq_score = min(50 + (np.log2(sample / 50) * 25), 100)
            return round((confidence_score * 0.6) + (freq_score * 0.4), 1)
        
        df['score'] = df.apply(calc_score, axis=1)
    
    # Rename columns for consistency
    if 'condition_count' in df.columns and 'total_days' not in df.columns:
        df['total_days'] = df['condition_count']
    
else:
    # Data is in combo format with 'bullish_pct' as percentage
    # Create outcome column
    df['outcome'] = df['directional_bias']
    
    # Calculate score
    if 'actionability_score' not in df.columns:
        def calc_score(row):
            edge = row['edge']
            confidence_score = min(edge * 2, 100)
            sample = row['total_days']
            if sample < 50:
                freq_score = (sample / 50) * 50
            else:
                freq_score = min(50 + (np.log2(sample / 50) * 25), 100)
            return round((confidence_score * 0.6) + (freq_score * 0.4), 1)
        
        df['score'] = df.apply(calc_score, axis=1)
    else:
        df['score'] = df['actionability_score']
    
    # Convert bullish_pct to confidence (decimal)
    if 'bullish_pct' in df.columns and 'confidence' not in df.columns:
        df['confidence'] = df['bullish_pct'] / 100

# ============================================================
# DISPLAY FUNCTION
# ============================================================

def display_patterns_by_complexity(df, title="INITIAL BALANCE"):
    print("\n" + "="*130)
    print(f"{title} PATTERNS")
    print("="*130)
    
    for feat_count in [1, 2, 3, 4]:
        patterns = df[df['feature_count'] == feat_count].sort_values('score', ascending=False)
        
        if len(patterns) == 0:
            continue
        
        bullish = patterns[patterns['outcome'] == 'bullish']
        bearish = patterns[patterns['outcome'] == 'bearish']
        
        print(f"\n{'-'*130}")
        print(f"{feat_count}-FEATURE PATTERNS")
        print(f"{'-'*130}")
        print(f"Total: {len(patterns):,} | Bullish: {len(bullish):,} | Bearish: {len(bearish):,}")
        
        # Show top 50 bullish
        print(f"\nTOP 50 BULLISH ({feat_count}-feature):")
        print(f"{'#':<3} {'Score':<6} {'Conf':<7} {'Edge':<7} {'Days':<6} {'Conditions':<80}")
        print("-"*130)
        
        for i, (idx, row) in enumerate(bullish.head(50).iterrows(), 1):
            # Build conditions based on format
            if USE_SMART_FORMAT:
                cond = f"{row['f1']}={row['v1']}"
                if feat_count >= 2:
                    cond += f", {row['f2']}={row['v2']}"
                if feat_count >= 3:
                    cond += f", {row['f3']}={row['v3']}"
                if feat_count >= 4 and 'f4' in row and pd.notna(row['f4']):
                    cond += f", {row['f4']}={row['v4']}"
            else:
                cond = row['features'].replace(' + ', ', ')
                values = row['values'].replace(' + ', ', ')
                cond = f"{cond} = {values}"
            
            cond = cond[:80]
            
            # Calculate metrics
            if USE_SMART_FORMAT:
                conf_pct = row['confidence'] * 100
                true_edge = (row['confidence'] - BASELINE_BULLISH) * 100
                days = row.get('condition_count', row.get('total_days', 0))
            else:
                conf_pct = row.get('bullish_pct', row['confidence'] * 100)
                true_edge = conf_pct - (BASELINE_BULLISH * 100)
                days = row['total_days']
            
            print(f"{i:<3} {row['score']:<6.1f} {conf_pct:<6.1f}% {true_edge:+6.1f}% {days:<6} {cond}")
        
        # Show top 50 bearish
        print(f"\nTOP 50 BEARISH ({feat_count}-feature):")
        print(f"{'#':<3} {'Score':<6} {'Conf':<7} {'Edge':<7} {'Days':<6} {'Conditions':<80}")
        print("-"*130)
        
        if len(bearish) > 0:
            for i, (idx, row) in enumerate(bearish.head(50).iterrows(), 1):
                # Build conditions
                if USE_SMART_FORMAT:
                    cond = f"{row['f1']}={row['v1']}"
                    if feat_count >= 2:
                        cond += f", {row['f2']}={row['v2']}"
                    if feat_count >= 3:
                        cond += f", {row['f3']}={row['v3']}"
                    if feat_count >= 4 and 'f4' in row and pd.notna(row['f4']):
                        cond += f", {row['f4']}={row['v4']}"
                else:
                    cond = row['features'].replace(' + ', ', ')
                    values = row['values'].replace(' + ', ', ')
                    cond = f"{cond} = {values}"
                
                cond = cond[:80]
                
                # Calculate metrics
                if USE_SMART_FORMAT:
                    conf_pct = row['confidence'] * 100
                    true_edge = (row['confidence'] - BASELINE_BEARISH) * 100
                    days = row.get('condition_count', row.get('total_days', 0))
                else:
                    bearish_pct = 100 - row.get('bullish_pct', row['confidence'] * 100)
                    conf_pct = bearish_pct
                    true_edge = bearish_pct - (BASELINE_BEARISH * 100)
                    days = row['total_days']
                
                print(f"{i:<3} {row['score']:<6.1f} {conf_pct:<6.1f}% {true_edge:+6.1f}% {days:<6} {cond}")
        else:
            print("  No bearish patterns")

# Display patterns
display_patterns_by_complexity(df, "INITIAL BALANCE")

# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("\n\n" + "="*130)
print("SUMMARY - PATTERN QUALITY BY COMPLEXITY")
print("="*130)

print(f"\nINITIAL BALANCE:")
print("-"*130)

for feat_count in [1, 2, 3, 4]:
    patterns = df[df['feature_count'] == feat_count]
    
    if len(patterns) == 0:
        continue
    
    bullish = patterns[patterns['outcome'] == 'bullish']
    bearish = patterns[patterns['outcome'] == 'bearish']
    
    print(f"\n  {feat_count}-Feature Patterns:")
    print(f"    Total: {len(patterns):,}")
    
    if len(bullish) > 0:
        print(f"    Bullish: {len(bullish):,} patterns")
        if USE_SMART_FORMAT:
            best_conf = bullish['confidence'].max() * 100
            true_edge = (bullish['confidence'].max() - BASELINE_BULLISH) * 100
            avg_sample = bullish.get('condition_count', bullish.get('total_days', 0)).mean()
        else:
            best_conf = bullish.get('bullish_pct', bullish['confidence'] * 100).max()
            true_edge = best_conf - (BASELINE_BULLISH * 100)
            avg_sample = bullish['total_days'].mean()
        
        print(f"      Best confidence: {best_conf:.1f}%")
        print(f"      Best true edge: {true_edge:+.1f}%")
        print(f"      Avg sample: {avg_sample:.0f} days")
    
    if len(bearish) > 0:
        print(f"    Bearish: {len(bearish):,} patterns")
        if USE_SMART_FORMAT:
            best_conf = bearish['confidence'].max() * 100
            true_edge = (bearish['confidence'].max() - BASELINE_BEARISH) * 100
            avg_sample = bearish.get('condition_count', bearish.get('total_days', 0)).mean()
        else:
            best_bearish_pct = (100 - bearish.get('bullish_pct', bearish['confidence'] * 100)).max()
            best_conf = best_bearish_pct
            true_edge = best_bearish_pct - (BASELINE_BEARISH * 100)
            avg_sample = bearish['total_days'].mean()
        
        print(f"      Best confidence: {best_conf:.1f}%")
        print(f"      Best true edge: {true_edge:+.1f}%")
        print(f"      Avg sample: {avg_sample:.0f} days")

# ============================================================
# TRADING RECOMMENDATIONS
# ============================================================

print("\n\n" + "="*130)
print("TRADING RECOMMENDATIONS")
print("="*130)

print("\n1. DAILY TRADING (High Frequency):")
print("   → Use 1-2 feature patterns")
print("   → Examples:")

# Get best 1-2 feature patterns
simple_bull = df[(df['feature_count'] <= 2) & (df['outcome'] == 'bullish')].sort_values('score', ascending=False).head(3)
simple_bear = df[(df['feature_count'] <= 2) & (df['outcome'] == 'bearish')].sort_values('score', ascending=False).head(3)

print("\n   IB Bullish:")
for i, (idx, row) in enumerate(simple_bull.iterrows(), 1):
    if USE_SMART_FORMAT:
        cond = f"{row['f1']}={row['v1']}"
        if row['feature_count'] >= 2:
            cond += f", {row['f2']}={row['v2']}"
        conf = row['confidence'] * 100
        days = row.get('condition_count', row.get('total_days', 0))
    else:
        cond = f"{row['features']} = {row['values']}"
        conf = row.get('bullish_pct', row['confidence'] * 100)
        days = row['total_days']
    
    print(f"     {i}. {cond[:70]} → {conf:.1f}% ({days} days)")

print("\n   IB Bearish:")
for i, (idx, row) in enumerate(simple_bear.iterrows(), 1):
    if USE_SMART_FORMAT:
        cond = f"{row['f1']}={row['v1']}"
        if row['feature_count'] >= 2:
            cond += f", {row['f2']}={row['v2']}"
        conf = row['confidence'] * 100
        days = row.get('condition_count', row.get('total_days', 0))
    else:
        cond = f"{row['features']} = {row['values']}"
        conf = 100 - row.get('bullish_pct', row['confidence'] * 100)
        days = row['total_days']
    
    print(f"     {i}. {cond[:70]} → {conf:.1f}% ({days} days)")

print("\n2. HIGH-CONFIDENCE SETUPS (Lower Frequency):")
print("   → Use 3+ feature patterns")
print("   → Only when all conditions align")
print("   → Examples:")

# Get best 3+ feature patterns
refined_bull = df[(df['feature_count'] >= 3) & (df['outcome'] == 'bullish')].sort_values('score', ascending=False).head(3)
refined_bear = df[(df['feature_count'] >= 3) & (df['outcome'] == 'bearish')].sort_values('score', ascending=False).head(3)

print("\n   IB Bullish:")
for i, (idx, row) in enumerate(refined_bull.iterrows(), 1):
    if USE_SMART_FORMAT:
        cond = f"{row['f1']}={row['v1']}, {row['f2']}={row['v2']}, {row['f3']}={row['v3']}"
        conf = row['confidence'] * 100
        days = row.get('condition_count', row.get('total_days', 0))
    else:
        cond = f"{row['features']} = {row['values']}"
        conf = row.get('bullish_pct', row['confidence'] * 100)
        days = row['total_days']
    
    print(f"     {i}. {cond[:70]} → {conf:.1f}% ({days} days)")

print("\n   IB Bearish:")
for i, (idx, row) in enumerate(refined_bear.iterrows(), 1):
    if USE_SMART_FORMAT:
        cond = f"{row['f1']}={row['v1']}, {row['f2']}={row['v2']}, {row['f3']}={row['v3']}"
        conf = row['confidence'] * 100
        days = row.get('condition_count', row.get('total_days', 0))
    else:
        cond = f"{row['features']} = {row['values']}"
        conf = 100 - row.get('bullish_pct', row['confidence'] * 100)
        days = row['total_days']
    
    print(f"     {i}. {cond[:70]} → {conf:.1f}% ({days} days)")

print("\n3. PATTERN COMPLEXITY:")
print("   - Simple patterns (1-2 features) = High frequency, good edge")
print("   - Complex patterns (3+ features) = Lower frequency, better edge")
print("   - Choose based on your trading style and risk tolerance")

print("\n" + "="*130)
print("COMPLETE")
print("="*130)

INITIAL BALANCE - TOP 50 PATTERNS BY COMPLEXITY

Loading IB pattern analysis results...
✓ Loading from combo files
  Loaded 2-feature: 902
  Loaded 3-feature: 899
  Loaded 4-feature: 1,599

✓ Total patterns loaded: 3,400

Baseline:
  54.6% bullish | 45.4% bearish


INITIAL BALANCE PATTERNS

----------------------------------------------------------------------------------------------------------------------------------
2-FEATURE PATTERNS
----------------------------------------------------------------------------------------------------------------------------------
Total: 902 | Bullish: 724 | Bearish: 178

TOP 50 BULLISH (2-feature):
#   Score  Conf    Edge    Days   Conditions                                                                      
----------------------------------------------------------------------------------------------------------------------------------
1   69.3   74.4  %  +19.8% 301    ny_orb_first_break, overnight_touched_s1 = high, True
2   69.1   74.3  %  +19


----------------------------------------------------------------------------------------------------------------------------------
3-FEATURE PATTERNS
----------------------------------------------------------------------------------------------------------------------------------
Total: 899 | Bullish: 556 | Bearish: 343

TOP 50 BULLISH (3-feature):
#   Score  Conf    Edge    Days   Conditions                                                                      
----------------------------------------------------------------------------------------------------------------------------------
1   73.6   78.5  %  +23.9% 191    ny_orb_first_break, ny_orb_direction, overnight_highest_pivot = high, bullish, R
2   73.3   77.7  %  +23.1% 265    ny_orb_first_break, ny_orb_direction, day_of_week = high, bullish, Monday
3   71.1   75.9  %  +21.3% 237    ny_orb_first_break, ny_orb_direction, overnight_touched_s1 = high, bullish, True
4   70.1   75.1  %  +20.5% 217    ny_orb_first_break, ny_orb_dir